## Customer Subscription Churn – EDA
## Executive Summary

This analysis identifies key drivers of churn in a subscription-based service.

Key risk factors include:
- Low weekly usage
- Multiple payment failures
- Early lifecycle churn (within first 6 months)

Retention efforts should prioritize improving early engagement and reducing billing friction.

## 1. Imports & Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style("whitegrid")

## 2. Load Data

In [ ]:
df = pd.read_csv('../data/raw/customer_subscriptions.csv')
df.head()

## 3. Data Overview

In [ ]:
print("Shape:", df.shape)

df.info()

df.describe()

df.isna().sum()

## 4. Data Cleaning

In [ ]:
# Convert date column
df['signup_date'] = pd.to_datetime(df['signup_date'])

# Create binary churn flag
df['churn_flag'] = df['churn'].map({'Yes': 1, 'No': 0})

# Overall churn rate
overall_churn = df['churn_flag'].mean()
print(f"Overall Churn Rate: {overall_churn:.2%}")

# Check outliers in usage
sns.boxplot(x=df['avg_weekly_usage_hours'])
plt.title("Usage Distribution")
plt.show()

## 5. Feature Engineering

In [ ]:
df['usage_level'] = pd.cut(
    df['avg_weekly_usage_hours'],
    bins=[0, 5, 10, df['avg_weekly_usage_hours'].max()],
    labels=['Low', 'Medium', 'High']
)

## 6. Exploratory Data Analysis (EDA)

In [ ]:
# Churn distribution
sns.countplot(x='churn', data=df)
plt.title('Churn Distribution')
plt.show()

# Usage vs churn
sns.boxplot(x='churn', y='avg_weekly_usage_hours', data=df)
plt.title('Usage vs Churn')
plt.show()

# Churn rate by plan
plan_churn = df.groupby('plan_type')['churn_flag'].mean().reset_index()

sns.barplot(x='plan_type', y='churn_flag', data=plan_churn)
plt.title('Churn Rate by Plan')
plt.ylabel('Churn Rate')
plt.show()

# Payment failures vs churn
sns.boxplot(x='churn', y='payment_failures', data=df)
plt.title('Payment Failures vs Churn')
plt.show()

## Key Findings

- Churned users show significantly lower weekly usage.
- Customers with more payment failures exhibit higher churn rates.
- Churn differs across subscription plans.
- A significant portion of churn occurs early in the customer lifecycle.

## Correlation Analysis

In [ ]:
plt.figure(figsize=(10,6))

corr = df.select_dtypes(include=['int64','float64']).corr()

sns.heatmap(corr, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

## Early Churn Analysis

In [ ]:
# Early churn defined as churn within first 6 months
early_churn = df[(df['tenure_months'] <= 6) & (df['churn_flag'] == 1)].shape[0]
total_churn = df[df['churn_flag'] == 1].shape[0]

print(f"Early Churn % among churned users: {early_churn/total_churn:.2%}")

## Save Cleaned Dataset

In [ ]:
df.to_csv('../data/processed/cleaned_customer_subscriptions.csv', index=False)

## Business Insights & Recommendations

### Insight 1 – Low Engagement Drives Churn

**What we observed:**  
Churned users show significantly lower weekly usage hours.
Correlation matrix indicates a negative relationship between usage and churn_flag.

**Why it happens:**  
Low product usage suggests customers are not integrating the product into their routine.
Without consistent engagement, perceived value declines.

**So what?**  
Churn appears to be behavior-driven rather than purely price-driven.
Engagement level is a strong early indicator of churn risk

**Recommendation:**  
Implement automated nudges and onboarding reinforcement for low-usage users.
Trigger automated nudges for low-usage users
Strengthen onboarding to encourage habit formation
Offer feature education emails during first 90 days

### Insight 2 – Early Lifecycle Vulnerability

**What we observed:**  
A significant percentage of churn occurs within the first 6 months.
Early churn rate among churned users is high.

**Why it happens:**  
Customers may not reach activation or value realization stage early enough.
If users fail to see value quickly, they disengage.

**So what?**  
The first 90–180 days represent the highest churn risk window.
Retention efforts should be lifecycle-focused.

**Recommendation:**  
Create structured onboarding program
Introduce milestone-based engagement targets
Assign early success metrics (activation KPIs)
Offer proactive support during first 3 months


### Insight 3 – Payment Failures Increase Churn

**What we observed:**  
Users with multiple payment failures have significantly higher churn rates.
Payment failures distribution skews higher among churned users

**Why it happens:**  
Billing friction causes frustration and reduces trust.
Some churn may not be intentional — it may be operational.

**So what?**  
Not all churn is behavioral — some is operational and preventable.

**Recommendation:**  
Implement automated payment retry logic
Send proactive payment reminders
Provide flexible payment options
Improve billing UX

### Insight 4 – Pricing Alone Does Not Guarantee Retention

**What we observed:**  
Churn varies across subscription plans.
Premium users do not necessarily show lower churn.

**Why it happens:**  
Higher price does not automatically mean higher perceived value.
Value communication may be insufficient.

**So what?**  
Retention depends more on engagement and perceived utility than pricing tier.

**Recommendation:**  
Improve premium value messaging
Highlight feature differentiation
Offer feature walkthroughs for premium users

## Strategic Summary

The analysis indicates that churn is predictable and primarily driven by:

1. Low product engagement
2. Early lifecycle vulnerability
3. Billing friction

These drivers are actionable.

The highest ROI opportunity lies in improving early onboarding engagement and reducing payment-related friction.